In [2]:
import numpy as pd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [26]:
# 1-8호선 데이터셋 불러오기
line_1to8 = pd.read_csv("../../data/raw/서울교통공사_1-8호선_승강기_가동현황.csv", encoding='cp949')

In [8]:
line_1to8.head()

,역코드,역명,승강기명,운행구간,설치위치,운행상태,승강기 구분
0,2723,면목,승강기)에스컬레이터-면목 내부 2호기,B3-B1,대합실,사용가능,ES
1,2723,면목,승강기)엘리베이터-면목 내부1,B1-B3-B4,상봉 방면4-4,사용가능,EV
2,2723,면목,승강기)엘리베이터-면목 내부2,B3-B4,사가정 방면5-2,사용가능,EV
3,2723,면목,승강기)엘리베이터-면목 외부3,B1-1F,3번 출입구,사용가능,EV
4,150,서울역(1),승강기)에스컬레이터-서울(1)역 4번출구 4호기,B1-1F,4번 출입구,사용가능,ES


In [28]:
# 9호선 엘레베이터, 에스컬레이터, 무빙워크 데이터셋 불러오기
line9_es = pd.read_excel("../../data/raw/서울메트로9호선_에스컬레이터_20250103.xlsx", header=3)
line9_ev = pd.read_excel("../../data/raw/서울메트로9호선_엘리베이터_20250103.xlsx", header=3)
line9_mw = pd.read_excel("../../data/raw/서울메트로9호선_무빙워크_20250103.xlsx", header=3)

/opt/anaconda3/lib/python3.13/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():
/opt/anaconda3/lib/python3.13/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():
/opt/anaconda3/lib/python3.13/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


In [29]:
line9_es.head(1)

,No,철도운영기관명,운영노선명,역명,관리번호,상하행구분,(근접)출입구번호,시작층(지상/지하),시작층(운행역층),시작층(상세위치),종료층(지상/지하),종료층(운행역층),종료층(상세위치),승강기 상태,승강기형폭,승강기 일련번호,데이터 기준일자,참고사항
0,1,서울9호선,9호선,개화역,1,상행,NaN,지상,1,(1F) 김포공항역 방향 승강장 6-4 출입문 앞,지상,2,(2F) 표 내는 곳 앞,운행,1200형,1804-297,20231218,NaN


In [30]:
line9_ev.head(1)

,No,철도운영기관명,운영노선명,역명,관리번호,(근접)출입구번호,상세위치,시작층(지상/지하),시작층(운행역층),종료층(지상/지하),종료층(운행역층),정원(인원수),정원(중량)(kg),승강기 상태,승강기 일련번호,데이터 기준일자,참고사항,Unnamed: 17
0,1,서울9호선,9호선,개화역,1,NaN,(1F) 하선승강장 6-4 근처\n(2F) 안전관리실 옆,지상,1,지상,2,15.0,1000,운행,0029-358,20230508,NaN,NaN


In [31]:
line9_mw.head(1)

,No,철도운영기관명,운영노선명,역명,관리번호,지상지하구분,역층,(근접) 출입구번호,시작(운행방향),시작(상세위치),종료(운행방향),종료(상세위치),승강기상태코드,승강기 일련번호,데이터 기준일자,참고사항
0,1,서울9호선,9호선,동작,1,지하,2,NaN,NaN,(B2) 9호선 환승통로 앞,NaN,(B2) 9호선 환승에스컬레이터 앞,운행,1804-622,20231219,NaN


In [32]:
# '승강기명' 컬럼 통일 (시작위치 + 관리번호)
line9_es['관리번호'] = line9_es['관리번호'].astype(str)
line9_es['승강기명'] = line9_es['시작층(상세위치)'] + " " + line9_es['관리번호'] + "호기"

line9_ev['관리번호'] = line9_ev['관리번호'].astype(str)            
line9_ev['승강기명'] = line9_ev['상세위치'] + " " + line9_ev['관리번호'] + "호기"

line9_mw['관리번호'] = line9_mw['관리번호'].astype(str)
line9_mw['승강기명'] = line9_mw['시작(상세위치)'] + " " + line9_mw['관리번호'] + "호기"

In [33]:
# 승강기 구분 컬럼 생성
line9_es['승강기 구분'] = 'ES'
line9_ev['승강기 구분'] = 'EV'
line9_mw['승강기 구분'] = 'MW'

In [34]:
# 1-8호선 데이터에 '운영노선명' 컬럼 생성
line_1to8['운영노선명']=''

In [35]:
# 공통 컬럼 추출하기
line9_es.columns.intersection(line9_ev.columns).intersection(line9_mw.columns)
common_cols = ['운영노선명', '역명', '승강기명', '승강기 구분']

In [36]:
# 공통 컬럼 추출하기
line9_es.columns.intersection(line9_ev.columns).intersection(line9_mw.columns)
common_cols = ['운영노선명', '역명', '승강기명', '승강기 구분']

In [37]:
# 1-8호선 데이터셋과 9호선 데이터셋 병합하기
master_df = pd.concat([line9_ev[common_cols], line9_es[common_cols], line9_mw[common_cols], line_1to8[common_cols]])
master_df.loc[master_df['역명']=='개화역', '역명'] = '개화'

In [41]:
master_df

,운영노선명,역명,승강기명,승강기 구분
0,9호선,개화,(1F) 하선승강장 6-4 근처\n(2F) 안전관리실 옆 1호기,EV
1,9호선,개화,(1F) 2번출구 옆\n(2F) 편의점 앞 2호기,EV
2,9호선,김포공항,(B1) 안전관리실 옆\n(B3) 상부본선 승강장 4-2 지점 2호기,EV
3,9호선,김포공항,(B1) 안전관리실 앞\n(B4) 하선 승강장 1-1 지점 3호기,EV
4,9호선,김포공항,(B1) 안전관리실 앞\n(B4) 하선 승강장 1-1 지점 4호기,EV
...,...,...,...,...
2806,,남위례,승강기)에스컬레이터-남위례 내부 5호기,ES
2807,,남위례,승강기)에스컬레이터-남위례 내부 6호기,ES
2808,,남위례,승강기)엘리베이터-남위례 내부 1호기,EV
2809,,남위례,승강기)엘리베이터-남위례 내부 2호기,EV


In [43]:
master_df.to_csv('master.csv', index=False)